<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Predicción del aterrizaje de la primera etapa del Falcon 9 de SpaceX**


## Extracción de datos web de los registros de lanzamientos de Falcon 9 y Falcon Heavy de Wikipedia


Tiempo Estimado Necesario: **40** minutos


En este laboratorio, realizarás una extracción de datos web para recopilar registros históricos de lanzamientos del Falcon 9 de una página de Wikipedia titulada `Lista de lanzamientos del Falcon 9 y Falcon Heavy`.

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


La primera etapa del Falcon 9 aterrizará con éxito.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Aquí se muestran varios ejemplos de aterrizajes fallidos:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


Más concretamente, los registros de lanzamiento se almacenan en una tabla HTML que se muestra a continuación:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objetivos
Extracción de datos de lanzamientos del Falcon 9 con `BeautifulSoup`:
- Extraer una tabla HTML con los registros de lanzamientos del Falcon 9 de Wikipedia
- Analizar la tabla y convertirla en un dataframe de Pandas


Primero importemos los paquetes necesarios para este laboratorio.


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [6]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

y le proporcionaremos algunas funciones auxiliares para que pueda procesar la tabla HTML extraída de la web.


In [7]:
def date_time(table_cells):
    """
    Esta función devuelve la fecha y la hora de la celda de la tabla HTML.
    Entrada: el elemento de una celda de datos de la tabla. Extrae la fila adicional.
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    Esta función devuelve la versión mejorada de la celda de la tabla HTML.
    Entrada: el elemento de una celda de datos de la tabla que extrae la fila adicional.
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    Esta función devuelve el estado de aterrizaje de la celda de la tabla HTML.
    Entrada: el elemento de una celda de datos de la tabla. Extrae la fila adicional.
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
   Esta función devuelve el estado de aterrizaje de la celda de la tabla HTML.
   Entrada: el elemento de una celda de datos de la tabla. Extrae la fila adicional.
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


Para mantener la coherencia en las tareas del laboratorio, se le pedirá que extraiga los datos de una instantánea de la página wiki `Lista de lanzamientos de Falcon 9 y Falcon Heavy`, `actualizada el 9 de junio de 2021.`

In [11]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

A continuación, solicita la página HTML desde la URL anterior y obtén un objeto `response`.

### TAREA 1: Solicitar la página Wiki de lanzamiento de Falcon9 desde su URL


Primero, vamos a realizar una solicitud HTTP GET para pedir la página HTML de inicio de Falcon9, como respuesta HTTP.


In [16]:
# Utiliza el método requests.get() con la URL estática y los encabezados proporcionados.
# Asigna la respuesta a un objeto.
response = requests.get(static_url)


Crea un objeto `BeautifulSoup` a partir de la respuesta HTML.


In [22]:
# Utilice BeautifulSoup() para crear un objeto BeautifulSoup a partir del contenido de un texto de respuesta.

soup = BeautifulSoup(response.text, 'html.parser')


Imprime el título de la página para verificar si el objeto `BeautifulSoup` se creó correctamente.


In [23]:
# Usar el atributo soup.title

print(soup.title)

None


### TAREA 2: Extraer todos los nombres de columnas/variables del encabezado de la tabla HTML


A continuación, queremos recopilar todos los nombres de columna relevantes del encabezado de la tabla HTML.


Primero, intentemos encontrar todas las tablas en la página wiki. Si necesitas refrescar tu memoria sobre `BeautifulSoup`, consulta el enlace de referencia externa al final de este laboratorio.


In [ ]:
# Usa la función `find_all` del objeto `BeautifulSoup`, con el tipo de elemento `table`.
# Asigna el resultado a una lista llamada `html_tables`.


A partir de la tercera tabla, nuestra tabla objetivo contiene los registros de lanzamiento reales.


In [5]:
# Imprimamos la tercera tabla y comprobemos su contenido.
first_launch_table = html_tables[2]
print(first_launch_table)

NameError: name 'html_tables' is not defined

Debería poder ver los nombres de las columnas incrustados en los elementos del encabezado de la tabla `<th>` de la siguiente manera:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


A continuación, solo necesitamos iterar a través de los elementos `<th>` y aplicar la función `extract_column_from_header()` proporcionada para extraer el nombre de la columna uno por uno.


In [ ]:
column_names = []

# Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name (`if name is not None and len(name) > 0`) into a list called column_names


Compruebe los nombres de las columnas extraídas.


In [ ]:
print(column_names)

## TAREA 3: Crear un marco de datos analizando las tablas HTML de lanzamiento


Crearemos un diccionario vacío con las claves de los nombres de las columnas extraídas en la tarea anterior. Posteriormente, este diccionario se convertirá en un dataframe de Pandas.

In [ ]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

A continuación, solo necesitamos rellenar el `launch_dict` con los registros de lanzamiento extraídos de las filas de la tabla.


Por lo general, las tablas HTML en las páginas Wiki suelen contener anotaciones inesperadas y otros tipos de ruido, como enlaces de referencia `B0004.1[8]`, valores faltantes `N/A [e]`, formato inconsistente, etc.


Para simplificar el proceso de análisis, a continuación proporcionamos un fragmento de código incompleto que le ayudará a completar el `launch_dict`. Complete el siguiente fragmento de código con las tareas pendientes (TODO) o bien, puede escribir su propia lógica para analizar todas las tablas de lanzamiento.


In [ ]:
extracted_row = 0
#Extract each table 
for table_number,table in enumerate(soup.find_all('table',"wikitable plainrowheaders collapsible")):
   # get table row 
    for rows in table.find_all("tr"):
        #check to see if first table heading is as number corresponding to launch a number 
        if rows.th:
            if rows.th.string:
                flight_number=rows.th.string.strip()
                flag=flight_number.isdigit()
        else:
            flag=False
        #get table element 
        row=rows.find_all('td')
        #if it is number save cells in a dictonary 
        if flag:
            extracted_row += 1
            # Flight Number value
            # TODO: Append the flight_number into launch_dict with key `Flight No.`
            #print(flight_number)
            datatimelist=date_time(row[0])
            
            # Date value
            # TODO: Append the date into launch_dict with key `Date`
            date = datatimelist[0].strip(',')
            #print(date)
            
            # Time value
            # TODO: Append the time into launch_dict with key `Time`
            time = datatimelist[1]
            #print(time)
              
            # Booster version
            # TODO: Append the bv into launch_dict with key `Version Booster`
            bv=booster_version(row[1])
            if not(bv):
                bv=row[1].a.string
            print(bv)
            
            # Launch Site
            # TODO: Append the bv into launch_dict with key `Launch Site`
            launch_site = row[2].a.string
            #print(launch_site)
            
            # Payload
            # TODO: Append the payload into launch_dict with key `Payload`
            payload = row[3].a.string
            #print(payload)
            
            # Payload Mass
            # TODO: Append the payload_mass into launch_dict with key `Payload mass`
            payload_mass = get_mass(row[4])
            #print(payload)
            
            # Orbit
            # TODO: Append the orbit into launch_dict with key `Orbit`
            orbit = row[5].a.string
            #print(orbit)
            
            # Customer
            # TODO: Append the customer into launch_dict with key `Customer`
            customer = row[6].a.string
            #print(customer)
            
            # Launch outcome
            # TODO: Append the launch_outcome into launch_dict with key `Launch outcome`
            launch_outcome = list(row[7].strings)[0]
            #print(launch_outcome)
            
            # Booster landing
            # TODO: Append the launch_outcome into launch_dict with key `Booster landing`
            booster_landing = landing_status(row[8])
            #print(booster_landing)
            

Una vez que hayas introducido los valores del registro de lanzamiento analizados en `launch_dict`, puedes crear un dataframe a partir de ellos.


In [ ]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

Ahora podemos exportarlo a un archivo CSV para la siguiente sección, pero para que las respuestas sean consistentes y en caso de que tengas dificultades para completar este laboratorio.

Los siguientes laboratorios utilizarán un conjunto de datos proporcionado para que cada uno sea independiente.


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
